# Online Retail Customer Segmentation (Retail Domain)

## Objectives
**RFM analysis** + ANN classifier for high-value segment prediction using UCI Online Retail.

## Business Context
Retailers segment customers for campaigns, loyalty tiers, and churn prevention.


In [ ]:
# Optional: install dependencies (uncomment if needed)
# !pip install -q numpy pandas matplotlib seaborn scikit-learn tensorflow requests yfinance

import warnings
warnings.filterwarnings("ignore")

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")
print("TensorFlow:", tf.__version__)


In [ ]:
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
# Fallback CSV mirror if xlsx fails
try:
    df = pd.read_excel(URL)
except Exception:
    URL2 = "https://raw.githubusercontent.com/plotly/datasets/master/online-retail.csv"
    df = pd.read_csv(URL2)
print(df.shape)
df.head()


In [ ]:
df = df.dropna(subset=["CustomerID", "InvoiceDate"])
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Total"] = df["Quantity"] * df["UnitPrice"]
snapshot = df["InvoiceDate"].max() + pd.Timedelta(days=1)
rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Total", "sum"),
)
rfm.head()


In [ ]:
sns.pairplot(rfm.reset_index(drop=True)[["Recency","Frequency","Monetary"]].sample(500, random_state=SEED))
plt.show()
# High value = top 20% monetary
rfm["HighValue"] = (rfm["Monetary"] >= rfm["Monetary"].quantile(0.8)).astype(int)


In [ ]:
X = rfm[["Recency","Frequency","Monetary"]].values.astype(np.float32)
y = rfm["HighValue"].values


In [ ]:
# Train / validation / test split (stratified for classification)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=SEED, stratify=y_temp
)  # ~70/15/15

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print("Train:", X_train_s.shape, "Val:", X_val_s.shape, "Test:", X_test_s.shape)


In [ ]:
model = models.Sequential([
    layers.Input(shape=(3,)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["AUC"])
model.summary()


In [ ]:
checkpoint = callbacks.ModelCheckpoint(
    "retail_rfm_best.keras", monitor="val_loss", save_best_only=True, verbose=1
)
early_stop = callbacks.EarlyStopping(
    monitor="val_loss", patience=15, restore_best_weights=True, verbose=1
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1
)
cb_list = [checkpoint, early_stop, reduce_lr]


In [ ]:
history = model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=80,
    batch_size=32,
    callbacks=cb_list,
    verbose=1,
)

# Loss curves
pd.DataFrame(history.history).plot(figsize=(10, 4))
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.show()

# Test metrics
y_prob = model.predict(X_test_s, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1:", f1_score(y_test, y_pred, zero_division=0))
try:
    print("ROC-AUC:", roc_auc_score(y_test, y_prob))
except Exception:
    pass
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix (Test)")
plt.ylabel("True")
plt.xlabel("Predicted")
plt.show()


In [ ]:
# Inference demo on a few test rows
loaded = keras.models.load_model("retail_rfm_best.keras")
sample_idx = np.arange(min(5, len(X_test_s)))
samples = X_test_s[sample_idx]
probs = loaded.predict(samples, verbose=0).ravel()
for i, p in zip(sample_idx, probs):
    print(f"Row {i} -> P(high value): {p:.4f}, predicted: {int(p >= 0.5)}, actual: {int(y_test[i])}")

import joblib
joblib.dump(scaler, "scaler.pkl")
print("Saved scaler.pkl and retail_rfm_best.keras")


## Deployment Notes

1. **Serving**: Export with `model.export("saved_model")` for TensorFlow Serving, or wrap `predict` in FastAPI/Flask.
2. **Preprocessing**: Always apply the **same** scaler/encoder fitted on training data (`scaler.pkl`).
3. **Monitoring**: Track input drift, latency, and prediction distribution on live traffic.
4. **Retraining**: Schedule periodic retrain when performance drops below SLA.
5. **Security**: Do not log PII; use HTTPS and auth on inference endpoints.
